# Analysis of Minor League Challenge Systems

In this file we will analyze the data we prepared in `data_prep.ipynb`.

## Questions to Pursue

1) What types of pitches and pitch locations are challenged most frequently?

2) How often are challenges successful?

3) Which batters / teams challenge most frequently?

4) Who uses their challenges most / least effectively? Could incorporate run values here

5) Can we create a challenge 'player card' visualization as a proof of concept?

## Current Blockers

1) It turns out that sz_top and sz_bottom don't align perfectly with the top and bottom of the strike zone. It would be helpful to definitiely determine whether a challenge will be successful to analyze 'missed opportunities'.

## Next Steps

1) Begin analysis with the questions above as a guide.

2) Look into using the zone column to determine whether a pitch is in the strike zone.

# Import Libraries

In [4]:
import numpy as np
import polars as pl

# Load Data

In [2]:
pbp = pl.read_csv('../data/export_data/aaa_pbp.csv')

In [3]:
pbp.head()

pitch_id,play_start_datetime,play_end_datetime,pitch_type,pitch_name,game_date,release_speed,release_pos_x,release_pos_y,release_pos_z,player_name,batter,pitcher,events,description,spin_dir,spin_rate_deprecated,break_angle_deprecated,break_length_deprecated,zone,des,game_type,stand,p_throws,home_team,away_team,type,hit_location,bb_type,balls,strikes,pfx_x,pfx_z,plate_x,plate_z,on_3b,on_2b,…,woba_denom,babip_value,iso_value,launch_speed_angle,at_bat_number,pitch_number,home_score,away_score,bat_score,fld_score,post_away_score,post_home_score,post_bat_score,post_fld_score,if_fielding_alignment,of_fielding_alignment,spin_axis,delta_home_win_exp,delta_run_exp,game_month,game_day,game_year,league_id,league_name,league_level_id,league_level_name,away_team_org_id,away_team_org_name,home_team_org_id,home_team_org_name,game_id,plate_appearance_id,lag_balls,lag_strikes,swing,challenge,challenge_successful
i64,str,str,str,str,str,f64,f64,f64,f64,str,i64,i64,str,str,f64,str,str,str,i64,str,str,str,str,str,str,str,f64,str,i64,i64,f64,f64,f64,f64,f64,f64,…,str,str,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,str,str,f64,str,str,i64,i64,i64,i64,str,i64,str,i64,str,i64,str,i64,i64,i64,i64,str,i64,str
0,"""2024-06-28T01:53:58.280""","""2024-06-28 01:54:02.789""","""CH""","""Changeup""","""2024-06-27""",86.6,-1.504636,50.000499,5.854567,"""Peter Lambert""",519303,663567,null,"""Elliot Soto strikes out on a f…",229.0,null,null,null,6,"""Elliot Soto strikes out on a f…","""R""","""R""","""R""","""ABQ""","""SL""","""C""",null,null,1,1,-6.941855,2.973652,0.753154,2.103768,null,null,…,null,null,null,null,34,2,6,1,1,6,1,6,1,6,null,null,229.0,null,null,6,27,2024,112,"""Pacific Coast League""",11,"""Triple-A""",108,"""Los Angeles Angels""",115,"""Colorado Rockies""",1036,22017,1,0,"""take""",0,"""no challenge"""
1,"""2024-09-18T00:19:15.010""","""2024-09-18 00:19:25.647""","""FF""","""Four-Seam Fastball""","""2024-09-17""",88.1,3.232269,50.004946,5.669619,"""Mason Fluharty""",666211,689254,null,"""Taylor Trammell called out on …",169.0,null,null,null,4,"""Taylor Trammell called out on …","""R""","""L""","""L""","""BUF""","""SWB""","""B""",null,null,1,0,-2.407521,4.213267,-0.760974,2.705338,null,null,…,null,null,null,null,58,1,3,3,3,3,3,3,3,3,null,null,169.0,null,null,9,17,2024,117,"""International League""",11,"""Triple-A""",147,"""New York Yankees""",141,"""Toronto Blue Jays""",807,22994,0,0,"""take""",0,"""no challenge"""
2,"""2024-07-25T03:06:15.107""","""2024-07-25 03:06:19.295""","""SL""","""Slider""","""2024-07-24""",85.0,0.673434,50.003952,5.825734,"""Blake Taylor""",680862,642130,null,"""Willie MacIver strikes out on …",288.0,null,null,null,13,"""Willie MacIver strikes out on …","""R""","""R""","""L""","""ABQ""","""RR""","""B""",null,null,2,2,-1.537159,-0.80345,-1.014508,0.475927,null,null,…,null,null,null,null,73,7,4,7,4,7,7,4,4,7,null,null,288.0,null,null,7,24,2024,112,"""Pacific Coast League""",11,"""Triple-A""",140,"""Texas Rangers""",115,"""Colorado Rockies""",943,66672,1,2,"""take""",0,"""no challenge"""
3,"""2024-08-03T22:40:47.492""","""2024-08-03 22:40:51.501""","""FF""","""Four-Seam Fastball""","""2024-08-03""",95.2,-0.00569,50.002266,6.428304,"""Braydon Fisher""",681508,680755,null,"""Mickey Gasper walks. Triston…",203.0,null,null,null,11,"""Mickey Gasper walks. Triston…","""R""","""L""","""R""","""WOR""","""BUF""","""B""",null,null,2,2,-3.876377,10.492538,-0.363394,4.554971,805367.0,671213.0,…,null,null,null,null,71,4,9,5,9,5,5,9,9,5,null,null,203.0,null,null,8,3,2024,117,"""International League""",11,"""Triple-A""",141,"""Toronto Blue Jays""",111,"""Boston Red Sox""",615,24923,1,2,"""take""",0,"""no challenge"""
4,"""2024-08-16T23:05:44.932""","""2024-08-16 23:05:51.595""","""CU""","""Curveball""","""2024-08-16""",74.2,-1.255317,50.000262,5.939749,"""Carlos Rodriguez""",682927,692230,null,"""Ronny Simon grounds out, first…",47.0,null,null,null,13,"""Ronny Simon grounds out, first…","""R""","""L""","""R""","""DUR""","""NAS""","""

# Classify Pitch as In or Outside of Strikezone

**Horizontal limits:** x is centered at 0 and measured in feet. Home plate is 17 inches wide, and the strike zone's horizontal limits include home plate plus the diameter of the baseball (2.94 inches) = 19.94 inches = 1.66 feet. Thus, the limits of the strike zone are +- 0.83.

**Vertical limits:** Right now we're using sz_top and sz_bot +/- the diameter of the ball, but as we'll see below it's not entirely consistent with ABS results.

In [9]:
ball_types = ['B', '*B', 'P']
pbp = (
    pbp
        .with_columns(
            pl.when(
                (np.abs(pl.col('plate_x')) < 0.83) &
                (pl.col('plate_z') >= (pl.col('sz_bot') - (2.94 / 12))) &
                (pl.col('plate_z') <= (pl.col('sz_top') + (2.94 / 12)))
            )
            .then(1)
            .otherwise(0)
            .alias('in_strike_zone')
            )
        .with_columns(
            # called a ball, but in the strike zone
            pl.when(
                (
                    (pl.col('type').is_in(ball_types)) &
                    (pl.col('in_strike_zone') == 1)
                ) |
                # called a strike, but not in the strike zone
                (
                    (pl.col('swing') == 'take') &
                    (pl.col('strikes') > pl.col('lag_strikes')) &
                    (pl.col('in_strike_zone') == 0)
                )
            )
            .then(1)
            .otherwise(0)
            .alias('missed_call')
        )
)

In [10]:
# Why are there missed calls on challenged pitches?
(
    pbp
        .filter(pl.col('challenge') == 1)
        .select(pl.col('missed_call').value_counts()).unnest('missed_call')
)

missed_call,count
i32,u32
1,110
0,821


In [13]:
a = (
    pbp
        .filter(
            (pl.col('challenge') == 1) &
            (pl.col('missed_call') == 1)
        )
        .with_columns([
            pl.col('sz_bot') - (2.94 / 12),
            pl.col('sz_top') + (2.94 / 12)
        ])
        .select(
            'balls',
            'strikes',
            'lag_balls',
            'lag_strikes',
            'plate_x',
            'plate_z',
            'sz_top',
            'sz_bot',
            'des'
        )
)
# first one: ABS determined it was a strike despite being slightly below the zone
# maybe slightly increase the width of the zone